In [1]:
from glob import glob
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from open_clip import get_tokenizer, create_model_and_transforms

model_name = 'hf-hub:timm/PE-Core-bigG-14-448'
model, _, preprocess = create_model_and_transforms(model_name)
tokenizer = get_tokenizer(model_name)

c:\Users\Bui Thien Nghia\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
txt = 'hello! am i cute?'
context_length = model.context_length
img_encode = model.visual

del model

In [4]:
img_encode = img_encode.eval().cuda()

In [5]:
class Keyframes(Dataset):
    def __init__(self, data):
        super().__init__()
        self.data = data

    
    def __len__(self):
        return len(self.data)
    

    def __getitem__(self, index):
        return self.data[index]

In [8]:
items = [preprocess(Image.open(path)) for path in glob(r'D:\AIC2025\kfs\Keyframes_K01\K01_V001\*.jpg')]
dataset = Keyframes(items)

dataset = DataLoader(dataset, pin_memory=True)

In [9]:
features = []
with torch.no_grad():
    for item in tqdm(dataset, desc='Computing features: '):
        try:
            features.append(F.normalize(img_encode(item.cuda()), dim=-1))
        except Exception as e:
            print(f'Error: {e}')
            break

Computing features: 100%|██████████| 409/409 [04:17<00:00,  1.59it/s]


In [13]:
features = torch.stack(features, dim=0)

In [18]:
features = features.squeeze(1)

In [22]:
print(features.nelement() * features.element_size())

2094080
